# Week 02

## TFIDF

Goal: compare TFIDF with BoW retrieval

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', None)

import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

from collections import Counter
import matplotlib.pyplot as plt

Importing the data:

In [2]:
# from https://www.microsoft.com/en-us/download/details.aspx?id=52419
data_folder = Path("WikiQACorpus/WikiQACorpus/")

train = pd.read_csv(data_folder/"WikiQA-train.txt", header=None, sep = r'\t', engine='python')

column_names = ["q", "a", "is_answer"]
train.columns = column_names

train.sort_values("q", inplace=True)
train

,q,a,is_answer
987,HOW MANY BROTHELS WERE THERE IN THE UNITED STATES IN 1840 'S,"Currently , Nevada is the only state to allow brothel prostitution , the terms of which are stipulated in the Nevada Revised Statutes .",0
991,HOW MANY BROTHELS WERE THERE IN THE UNITED STATES IN 1840 'S,"Street prostitution , `` pandering , '' and living off of the proceeds of a prostitute remain illegal under Nevada law , as elsewhere in the country .",0
984,HOW MANY BROTHELS WERE THERE IN THE UNITED STATES IN 1840 'S,"The regulation of prostitution in the United States is not among the enumerated powers of the federal government ; Under the Tenth Amendment to the United States Constitution , it is therefore exclusively the domain of the states to permit , prohibit , or otherwise regulate commercial sex , except insofar as Congress may regulate it as part of interstate commerce with laws like the Mann Act .",0
990,HOW MANY BROTHELS WERE THERE IN THE UNITED STATES IN 1840 'S,"The other counties theoretically allow brothel prostitution , but some of these counties currently have no active brothels .",0
989,HOW MANY BROTHELS WERE THERE IN THE UNITED STATES IN 1840 'S,"All forms of prostitution are illegal in Clark County ( which contains the Las Vegas–Paradise metropolitan area ) , Washoe County ( which contains Reno ) , Carson City , Douglas County , and Lincoln County .",0
...,...,...,...
7295,who wrote white christmas,"According to the Guinness Book of World Records , the version sung by Bing Crosby is the best-selling single of all time , with estimated sales in excess of 50 million copies worldwide .",0
7296,who wrote white christmas,Accounts vary as to when and where Berlin wrote the song .,0
7297,who wrote white christmas,"One story is that he wrote it in 1940 , in warm La Quinta , California , while staying at the La Quinta Hotel , a frequent Hollywood retreat also favored by writer-producer Frank Capra , although the Arizona Biltmore also claims the song was written there .",0
7298,who wrote white christmas,"He often stayed up all night writing — he told his secretary , `` Grab your pen and take down this song .",0


Add a counter within each group of questions (group size):

In [3]:
train['q_count'] = train.groupby('q')['q'].transform('count')
train.query('q_count > 1')

,q,a,is_answer,q_count
987,HOW MANY BROTHELS WERE THERE IN THE UNITED STATES IN 1840 'S,"Currently , Nevada is the only state to allow brothel prostitution , the terms of which are stipulated in the Nevada Revised Statutes .",0,9
991,HOW MANY BROTHELS WERE THERE IN THE UNITED STATES IN 1840 'S,"Street prostitution , `` pandering , '' and living off of the proceeds of a prostitute remain illegal under Nevada law , as elsewhere in the country .",0,9
984,HOW MANY BROTHELS WERE THERE IN THE UNITED STATES IN 1840 'S,"The regulation of prostitution in the United States is not among the enumerated powers of the federal government ; Under the Tenth Amendment to the United States Constitution , it is therefore exclusively the domain of the states to permit , prohibit , or otherwise regulate commercial sex , except insofar as Congress may regulate it as part of interstate commerce with laws like the Mann Act .",0,9
990,HOW MANY BROTHELS WERE THERE IN THE UNITED STATES IN 1840 'S,"The other counties theoretically allow brothel prostitution , but some of these counties currently have no active brothels .",0,9
989,HOW MANY BROTHELS WERE THERE IN THE UNITED STATES IN 1840 'S,"All forms of prostitution are illegal in Clark County ( which contains the Las Vegas–Paradise metropolitan area ) , Washoe County ( which contains Reno ) , Carson City , Douglas County , and Lincoln County .",0,9
...,...,...,...,...
7295,who wrote white christmas,"According to the Guinness Book of World Records , the version sung by Bing Crosby is the best-selling single of all time , with estimated sales in excess of 50 million copies worldwide .",0,7
7296,who wrote white christmas,Accounts vary as to when and where Berlin wrote the song .,0,7
7297,who wrote white christmas,"One story is that he wrote it in 1940 , in warm La Quinta , California , while staying at the La Quinta Hotel , a frequent Hollywood retreat also favored by writer-producer Frank Capra , although the Arizona Biltmore also claims the song was written there .",0,7
7298,who wrote white christmas,"He often stayed up all night writing — he told his secretary , `` Grab your pen and take down this song .",0,7


Exclude:
 - Questions with too less answers (less than *min_q*)
 - Questions that have no true answer

In [4]:
def filter_data(df, min_q = 5):
    df['q_count'] = df.groupby('q')['q'].transform('count')
    df['has_correct_answer'] = df.groupby('q')['is_answer'].transform('max')
    return df.query("q_count >= @min_q").query("has_correct_answer == 1")

print(f"before: {train.shape}")
train = filter_data(train)
print(f"after: {train.shape}")

before: (20360, 4)
after: (8108, 5)


In [5]:
train.describe()

,is_answer,q_count,has_correct_answer
count,8108.000000,8108.000000,8108.0
mean,0.101505,14.704243,1.0
std,0.302014,6.458040,0.0
min,0.000000,5.000000,1.0
25%,0.000000,10.000000,1.0
50%,0.000000,14.000000,1.0
75%,0.000000,19.000000,1.0
max,1.000000,30.000000,1.0


Looks okay.

Data preprocessing:
 - your choice. Candidate operations include stopword removal, lowercasing, remove answers that have only one or two words after stopword removal, etc.

In [6]:
# your turn

stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

def get_len_0_token_bool(df, col):
    return [elem == 0 for elem in map(len, df[col])]

def drop_rows(df, bool_vec):
    print(f" - dropping {sum(bool_vec)} rows.")
    keep_vec = [not elem for elem in bool_vec]
    return df.loc[keep_vec, :]

def data_clean(text, stop_words = stop_words):
    """ cleans a text string by lowercasing, removing stopwords and non-words """
    clean = lambda text: [ps.stem(word) for word in word_tokenize(text.lower()) if word not in stop_words]
    cleaned = clean(text)
    cleaned = [re.sub(r'[^\w\s]', '', word) for word in cleaned]
    return " ".join([word for word in cleaned if word != ""])

def data_preprocess(df):
    """ creates two more columns for Q/A with cleaned and tokenized word lists"""
    df = df.copy()

    for c in ["q", "a"]:
        col = c + "_tokenized"
        print(f"cleaning {col}:")

        df[col] = df[c].apply(data_clean)
        bool_vec = get_len_0_token_bool(df, col)
        df = drop_rows(df, bool_vec)

    return df.reset_index(drop=True)

In [7]:
train = data_preprocess(train)

cleaning q_tokenized:
 - dropping 0 rows.
cleaning a_tokenized:
 - dropping 1 rows.


In [8]:
train.head(3)

,q,a,is_answer,q_count,has_correct_answer,q_tokenized,a_tokenized
0,How Do You Get Hepatitis C,"In some cases , those with cirrhosis will go on to develop liver failure , liver cancer or life-threatening esophageal and gastric varices .",0,13,1,get hepat c,case cirrhosi go develop liver failur liver cancer lifethreaten esophag gastric varic
1,How Do You Get Hepatitis C,"Hepatitis C is an infectious disease affecting primarily the liver , caused by the hepatitis C virus ( HCV ) .",0,13,1,get hepat c,hepat c infecti diseas affect primarili liver caus hepat c viru hcv
2,How Do You Get Hepatitis C,"The infection is often asymptomatic , but chronic infection can lead to scarring of the liver and ultimately to cirrhosis , which is generally apparent after many years .",0,13,1,get hepat c,infect often asymptomat chronic infect lead scar liver ultim cirrhosi gener appar mani year


In [9]:
print(f"number of documents {train.shape[0]}")
print(f"number of words {len(set(train.a_tokenized.apply(lambda x: x.split()).explode()))}")

number of documents 8107
number of words 14460


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer()
X_answers = vectorizer.fit_transform(train.a_tokenized)
vectorizer.get_feature_names_out()

array(['00000000001', '00001', '000590', ..., 'ሴም', '一圖勝萬言', '劉安'],
      shape=(14425,), dtype=object)

In [11]:
print(f"TF-IDF matrix shape: {X_answers.shape}")
print(f"Number of answers: {X_answers.shape[0]}, Vocabulary size: {X_answers.shape[1]}")

TF-IDF matrix shape: (8107, 14425)
Number of answers: 8107, Vocabulary size: 14425


In [12]:
# Function to search for the best matching answer
def find_best_answers(question_tokens, vectorizer, top_k=5):
    """
    Find top-k best matching answers for a question.
    
    Args:
        question_tokens: list of tokens from a question
        vectorizer: fitted vectorizer (TfidfVectorizer or CountVectorizer)
        top_k: number of top answers to return
    """


    # Convert question tokens to TF-IDF representation
    X_question = vectorizer.transform([question_tokens])
    
    # Calculate cosine similarity with all answers
    similarities = cosine_similarity(X_question, X_answers)[0]
    
    # Get top-k indices
    top_indices = np.argsort(similarities)[::-1][:top_k]
    top_similarities = similarities[top_indices]
    
    return top_indices, top_similarities

In [13]:
top_indices, top_similarities = find_best_answers(train.q_tokenized.iloc[0], vectorizer=vectorizer, top_k=5)
print(f"Question: {train.q_tokenized.iloc[0]}")
print("Top-3 matching answers:")
for i, idx in enumerate(top_indices[:3]):
    print(f"  {i+1}. {train.a_tokenized.iloc[idx]}")

Question: get hepat c
Top-3 matching answers:
  1. hepat c infecti diseas affect primarili liver caus hepat c viru hcv
  2. agent requir help artist get job
  3. rare 3 chanc get gene


In [14]:
# Evaluate how often the correct answer is in top-k
def evaluate_retrieval(df, vectorizer, model_name, top_k_values=[5, 10]):
    """
    Evaluate retrieval performance for all unique questions
    """
    unique_questions = df['q'].unique()
    results = {k: [] for k in top_k_values}
    max_k = max(top_k_values)  # ← Add this
    
    for question in unique_questions:
        # Get the question tokens and correct answer index
        q_data = df[df['q'] == question]
        q_tokens = q_data['q_tokenized'].iloc[0]
        correct_idx = q_data[q_data['is_answer'] == 1].index[0]
        
        # Find best answers
        top_indices, similarities = find_best_answers(q_tokens, vectorizer=vectorizer, top_k=max_k)
        
        # Check for each top-k
        for k in top_k_values:
            is_correct = correct_idx in top_indices[:k]
            results[k].append(is_correct)
    
    print(f"\n{model_name} Results:")
    for k in top_k_values:
        accuracy = np.mean(results[k])
        print(f"  Top-{k}: {accuracy:.2%}")
    
    return results

In [15]:
# Run evaluation
tfidf_results = evaluate_retrieval(train, vectorizer, "TF-IDF", [5, 10])


TF-IDF Results:
  Top-5: 50.95%
  Top-10: 62.92%


In [16]:
# same for count vectorizer
count_vectorizer = CountVectorizer()
X_answers_count = count_vectorizer.fit_transform(train.a_tokenized)
count_vectorizer.get_feature_names_out()

array(['00000000001', '00001', '000590', ..., 'ሴም', '一圖勝萬言', '劉安'],
      shape=(14425,), dtype=object)

In [17]:
count_vectorizer_results = evaluate_retrieval(train, count_vectorizer, "Count Vectorizer", [5, 10])


Count Vectorizer Results:
  Top-5: 46.42%
  Top-10: 58.83%
